# 01 — Data Cleaning v2
**Input:** `DATA.xlsx` → **Output:** `DATA_cleaned_v2.xlsx`

## What's new vs v1
- Fixes ALL systematic anonymisation corruption (11,000+ `ue` tokens, 8,000+ `Cutomer`, 5,000+ `uLL`, 4,600+ broken `-ue` suffixes)
- Extracts `se_flag` and `regulatory_flag` as clean binary features from Notes
- Adds `desc2_clean_bert` as a separate cleaned text stream
- Adds `desc_diff_len` feature (D2 longer than D1 = case was revised)
- Fixes digit-substitution corruption in Description 1 (SpO0larm, t0uchscreen, w0rking)

In [1]:
# !pip install pyspellchecker openpyxl spacy
# !python -m spacy download en_core_web_sm
import pandas as pd, numpy as np, re
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from spellchecker import SpellChecker
nlp = spacy.load('en_core_web_sm', disable=['parser','ner'])
print('Imports OK.')

/home/others/CS60078_P8/my_dsl_project/venv/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Imports OK.


In [2]:
INPUT_PATH  = 'DATA.xlsx'
SHEET_NAME  = 'Data'
OUTPUT_PATH = 'DATA_cleaned_v2.xlsx'

df = pd.read_excel(INPUT_PATH, sheet_name=SHEET_NAME)
print(f'Loaded {len(df):,} rows x {len(df.columns)} cols')
print('Important Event:', df['Important Event'].value_counts(dropna=False).to_dict())
print('Missing:', df.isnull().sum().to_dict())

Loaded 3,999 rows x 7 cols
Important Event: {'No': 3849, 'Yes': 133, 'Unknown': 12, nan: 5}
Missing: {'ID': 0, 'Indicator Flag': 3, 'Description 1': 0, 'Description 2': 0, 'Notes': 824, 'Date': 0, 'Important Event': 5}


In [3]:
DOMAIN_WORDS = {
    'defibrillator','defib','aed','ecg','ekg','ventilator','fluoroscopy','fluoroscopic',
    'xray','ultrasound','sonographer','monitor','biomed','biomedical','touchscreen',
    'touchpad','footswitch','paddles','paddle','leadwire','leadwires','pcb','hdd','ssd',
    'lcd','startup','bootup','selftest','blower','manifold','spo2','nibp','etco2',
    'ntf','fco','rse','pfs','assy','reboot','recalibration','calibration','defibrillation',
    'cardioversion','pacing','arrhythmia','tachycardia','bradycardia','waveform','chirp',
    'fleuvision','alarms','alarming','malfunction','malfunctioning','intermittent',
    'unresponsive','ops','cath','fluoro','bpm','pacemaker','tidal','biphasic','monophasic',
    'stent','catheter','pci','ventricular','atrial','fibrillation','cardiac',
}
spell = SpellChecker()
spell.word_frequency.load_words(DOMAIN_WORDS)
print(f'Domain dictionary: {len(DOMAIN_WORDS)} terms')

Domain dictionary: 71 terms


In [4]:
# ── COMPREHENSIVE CORRUPTION FIX TABLE ────────────────────────────────────
# Built from corpus analysis — covers all systematic anonymisation patterns

# 1. Words where final 's' was removed (appeared as -ue suffix)
UE_SUFFIX_FIXES = [
    # (broken_pattern, correct_word)
    (r'\bfailue\b',    'failure'),  (r'\bFailue\b',    'Failure'),
    (r'\bcaue\b',      'cause'),    (r'\bCaue\b',      'Cause'),
    (r'\bmodue\b',     'module'),   (r'\bModue\b',     'Module'),
    (r'\bbecaue\b',    'because'),  (r'\bBecaue\b',    'Because'),
    (r'\bpressue\b',   'pressure'), (r'\bPressue\b',   'Pressure'),
    (r'\bprocedue\b',  'procedure'),(r'\bProcedue\b',  'Procedure'),
    (r'\beuposue\b',   'exposure'), (r'\bEuposue\b',   'Exposure'),
    (r'\bfue\b',       'fuse'),     (r'\bFue\b',       'Fuse'),
    (r'\btue\b',       'true'),     (r'\bTue\b',       'True'),
    (r'\bisue\b',      'issue'),    (r'\bIsue\b',       'Issue'),
    (r'\blosue\b',     'closure'),
    (r'\bdiscloue\b',  'disclosure'),
    (r'\bencloue\b',   'enclosure'),
]

# 2. uLL = error/fault code label in structured notes
# 'uLL#' means 'Error Code #' or 'Fault Log #'
ULL_FIXES = [
    (r'\buLL#',           'Error Code:'),
    (r'\buLL:\s*',        'Error: '),
    (r'\buLLcode\b',      'error code'),
    (r'\buLLcodes\b',     'error codes'),
    (r'\buLLmessage\b',   'error message'),
    (r'\buLLmessages\b',  'error messages'),
    (r'\buLLlog\b',       'error log'),
    (r'\buLLlogs\b',      'error logs'),
    (r'\buLLmsg\b',       'error message'),
    (r'\buLLreport\b',    'error report'),
    (r'\buLLis\b',        'error is'),
    (r'\buLLwas\b',       'error was'),
    (r'\buLLand\b',       'error and'),
    (r'\buLLon\b',        'error on'),
    (r'\buLLin\b',        'error in'),
    (r'\buLLof\b',        'error of'),
    (r'\buLLwith\b',      'error with'),
    (r'\buLLwhen\b',      'error when'),
    (r'\buLLoccured\b',   'error occurred'),
    (r'\buLLduing\b',     'error during'),
    (r'\buLL2\d+',        'error '),
    (r'\buLL\b',          'error'),
]

# 3. Digit-0 substitutions in Description 1
# Systematic: certain letters replaced with 0 in the anonymisation
DIGIT_SUB_FIXES_DESC = [
    (r'\bSpO0',        'SpO2'),          # SpO0larm -> SpO2 alarm
    (r'\bt0uch',       'touch'),          # t0uchscreen -> touchscreen
    (r'\bw0rk',        'work'),           # w0rking -> working
    (r'\b0ifferent',   'different'),
    (r'\b0isplay',     'display'),
    (r'\b0etect',      'detect'),
    (r'\b0evice',      'device'),
    (r'\bCO0e',        'CO2e'),           # CO0ebreathing -> CO2e
    (r'\bDec0\b',      'Deco'),
    (r'\b0ec0\b',      'deco'),
    (r'\b0DEC\b',      'ODEC'),
]

# 4. Structured field broken words (same as v1 but extended)
STRUCTURED_FIXES = [
    (r'\bCutomer\b',      'Customer'),
    (r'\bcutomer\b',      'customer'),
    (r'\bEupected\b',     'Expected'),
    (r'\bueupected\b',    'unexpected'),
    (r'\bUeupected\b',    'Unexpected'),
    (r'\bResuts\b',       'Results'),
    (r'\bResoluion\b',    'Resolution'),
    (r'\btouhscreen\b',   'touchscreen'),
    (r'\bFaction\b',      'Function'),
    (r'\bfuction\b',      'function'),
    (r'\bFuction\b',      'Function'),
    (r'\bour Impact\b',   'User Impact'),
    (r'\bRetuned\b',      'Returned'),
    (r'\bretuned\b',      'returned'),
    (r'\bFollow u\b',     'Follow up'),
    (r'\bstart-u\b',      'start-up'),
    (r'\bu-ray\b',        'X-ray'),
    (r'\bNuber\b',        'Number'),
    (r'\bnuber\b',        'number'),
    (r'\brequed\b',       'required'),
    (r'\bRequed\b',       'Required'),
    (r'\buing\b',         'using'),
    (r'\bUing\b',         'Using'),
    (r'\bcoud\b',         'could'),
    (r'\bCoud\b',         'Could'),
    (r'\bwhee\b',         'where'),
    (r'\bclincal\b',      'clinical'),
]

print(f'Fix tables: {len(UE_SUFFIX_FIXES)} ue-suffix, {len(ULL_FIXES)} uLL, {len(DIGIT_SUB_FIXES_DESC)} digit-sub, {len(STRUCTURED_FIXES)} structured')

Fix tables: 23 ue-suffix, 22 uLL, 11 digit-sub, 27 structured


In [5]:
def apply_fixes(text, fix_list):
    if pd.isna(text): return text
    t = str(text)
    for pat, repl in fix_list:
        t = re.sub(pat, repl, t)
    return t

def remove_dataset_noise(text):
    """Remove anonymisation artifacts from Description 1."""
    if pd.isna(text): return text
    t = str(text)
    # Fix digit-0 substitutions first (before zero-removal)
    t = apply_fixes(t, DIGIT_SUB_FIXES_DESC)
    # 2+ leading zeros fused to a word  00he -> he
    t = re.sub(r'\b0{2,}([a-zA-Z])', r'\1', t)
    # Long zero-padded IDs
    t = re.sub(r'\b0{5,}[0-9a-zA-Z\-]*', '', t)
    t = re.sub(r'[0-9]*0{5,}\b', '', t)
    # Workflow code prefixes: M0A / u00/  DFM0  QAA0
    t = re.sub(r'^[A-Z0-9]{2,10}[:/][A-Z0-9/]*\s*', '', t)
    t = re.sub(r'^[A-Z]{1,5}\d+[/\s]', '', t)
    # Standalone u token
    t = re.sub(r'(?<![a-zA-Z])\bu\b(?![a-zA-Z])', '', t)
    # Digit-corrupted start of word  0ifferent -> ifferent
    t = re.sub(r'\b0([a-zA-Z]+)', r'\1', t)
    # Trailing noise
    t = re.sub(r'[\-_/]+\s*$', '', t)
    return re.sub(r'\s{2,}', ' ', t).strip()

def normalise_text(text):
    if pd.isna(text): return text
    t = str(text).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s{2,}', ' ', t).strip()

def correct_spelling(text):
    if pd.isna(text) or not str(text).strip(): return text
    words, out = str(text).split(), []
    for w in words:
        if len(w) <= 2 or w in DOMAIN_WORDS or not w.isalpha(): out.append(w)
        else:
            fixed = spell.correction(w)
            out.append(fixed if fixed else w)
    return ' '.join(out)

EXTRA_STOPS = {'unit','device','equipment','issue','problem','report','customer',
               'service','call','case','request','use','used','new','old',
               'model','number','na','nan','work','check','need','get','got'}
ALL_STOPS = STOP_WORDS | EXTRA_STOPS

def lemmatize_remove_stops(text):
    if pd.isna(text) or not str(text).strip(): return text
    doc = nlp(str(text))
    return ' '.join([t.lemma_ for t in doc
                     if t.lemma_ not in ALL_STOPS and len(t.lemma_) > 2 and t.is_alpha])

print('Cleaning functions defined.')

Cleaning functions defined.


In [6]:
# ── Clean Description 1 (4-stage pipeline) ───────────────────────────────
print('Cleaning Description 1  (~2 min)...')
df['_s1'] = df['Description 1'].apply(remove_dataset_noise)
df['_s2'] = df['_s1'].apply(normalise_text)
df['_s3'] = df['_s2'].apply(correct_spelling)
df['desc1_clean_bert'] = df['_s3']
df['desc1_clean_lda']  = df['_s3'].apply(lemmatize_remove_stops)
df.drop(columns=['_s1','_s2','_s3'], inplace=True)
print('Description 1 done.')

# ── Clean Description 2 (NEW) ─────────────────────────────────────────────
print('Cleaning Description 2...')
df['_s1'] = df['Description 2'].apply(remove_dataset_noise)
df['_s2'] = df['_s1'].apply(normalise_text)
df['_s3'] = df['_s2'].apply(correct_spelling)
df['desc2_clean_bert'] = df['_s3']
df.drop(columns=['_s1','_s2','_s3'], inplace=True)
print('Description 2 done.')

Cleaning Description 1  (~2 min)...
Description 1 done.
Cleaning Description 2...
Description 2 done.


In [7]:
# ── Clean Notes with comprehensive fix table ──────────────────────────────
def clean_notes_v2(text):
    """V2: applies ALL fix tables before section extraction."""
    if pd.isna(text): return text
    t = str(text)
    # Step 1: Remove zero-padded IDs and work order headers
    t = re.sub(r'0{10,}1O##:\s*[0-9]+', '', t)
    t = re.sub(r'\b0{5,}[0-9a-zA-Z\-]*', '', t)
    t = re.sub(r'[0-9]*0{5,}\b', '', t)
    # Step 2: Fix uLL tokens (error codes) BEFORE standalone-u removal
    t = apply_fixes(t, ULL_FIXES)
    # Step 3: Fix -ue suffix words (missing s)
    t = apply_fixes(t, UE_SUFFIX_FIXES)
    # Step 4: Fix structured field broken words
    t = apply_fixes(t, STRUCTURED_FIXES)
    # Step 5: Remove remaining standalone u tokens
    t = re.sub(r'(?<![a-zA-Z])\bu\b(?![a-zA-Z])', '', t)
    # Step 6: Clean whitespace
    t = re.sub(r'[ \t]{2,}', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    return t.strip()

def extract_section(text, start_pats, end_pats):
    if pd.isna(text): return np.nan
    t = str(text)
    m = re.search('|'.join(start_pats), t, re.IGNORECASE)
    if not m: return np.nan
    after = t[m.end():]
    m2 = re.search('|'.join(end_pats), after, re.IGNORECASE)
    sec = after[:m2.start()].strip() if m2 else after.strip()
    return sec if len(sec) > 5 else np.nan

print('Cleaning Notes  (comprehensive v2)...')
df['notes_clean'] = df['Notes'].apply(clean_notes_v2)

df['notes_problem_desc'] = df['notes_clean'].apply(lambda x: extract_section(x,
    [r'Expected and [Uu]nexpected [Rr]esults?\s*:', r'Problem Description'],
    [r'User Impact', r'Patient Impact', r'SE\s*[:\-]', r'-{10,}']))
df['notes_diagnostic']   = df['notes_clean'].apply(lambda x: extract_section(x,
    [r'Diagnostic performed', r'Diagnos', r'Root [Cc]ause'],
    [r'Resolution', r'Action [Tt]aken', r'-{10,}']))
df['notes_resolution']   = df['notes_clean'].apply(lambda x: extract_section(x,
    [r'Resolution\s*:', r'Action [Tt]aken'],
    [r'-{10,}', r'Internal Remark', r'Follow']))

print(f'notes_problem_desc: {df["notes_problem_desc"].notna().sum()} rows')
print(f'notes_diagnostic:   {df["notes_diagnostic"].notna().sum()} rows')
print(f'notes_resolution:   {df["notes_resolution"].notna().sum()} rows')

Cleaning Notes  (comprehensive v2)...
notes_problem_desc: 2707 rows
notes_diagnostic:   2256 rows
notes_resolution:   1958 rows


In [8]:
# ── Extract SE flag and Regulatory flag (NEW) ─────────────────────────────
# SE = Serious Event flag set by the engineer in structured notes
# This is one of the strongest predictors — engineer's own judgment
def extract_se_flag(text):
    """
    SE field values: 'SE unknown'=0, 'SE N/A'=0, 'SE an'=0 (=None),
                     'SE Yes'=1, 'SE No'=0
    Returns: 1 (serious), 0 (not serious), NaN (not found)
    """
    if pd.isna(text): return np.nan
    t = str(text)
    # Match: SE followed by Yes/No/unknown/N/A/an/none
    m = re.search(r'\bSE\s+(Yes|No|unknown|N/A|an\b|none|None)', t, re.I)
    if not m: return np.nan
    val = m.group(1).lower()
    if val == 'yes': return 1
    return 0

def extract_regulatory_flag(text):
    """
    Regulatory Questions: Yes/No or No/No
    First value = regulatory question answered Yes
    """
    if pd.isna(text): return 0
    t = str(text)
    m = re.search(r'Regulatory\s+Questions?\s*[:\-]?\s*(Yes|No)', t, re.I)
    if m and m.group(1).lower() == 'yes': return 1
    return 0

df['se_flag']         = df['Notes'].apply(extract_se_flag)
df['regulatory_flag'] = df['Notes'].apply(extract_regulatory_flag)

print('SE flag distribution:')
print(df['se_flag'].value_counts(dropna=False))
print()
print('Regulatory flag distribution:')
print(df['regulatory_flag'].value_counts(dropna=False))
print()
# Validate: cross with Important Event
print('SE flag vs Important Event:')
print(pd.crosstab(df['se_flag'].fillna('unknown'), df['Important Event']))

SE flag distribution:
se_flag
NaN    3761
0.0     238
Name: count, dtype: int64

Regulatory flag distribution:
regulatory_flag
0    3999
Name: count, dtype: int64

SE flag vs Important Event:
Important Event    No  Unknown  Yes
se_flag                            
0.0               238        0    0
unknown          3611       12  133


In [9]:
# ── Feature Engineering ───────────────────────────────────────────────────
df['target'] = df['Important Event'].map({'Yes': 1, 'No': 0})
df['hour_of_day']      = df['Date'].dt.hour
df['day_of_week']      = df['Date'].dt.dayofweek
df['is_weekend']       = (df['day_of_week'] >= 5).astype(int)
df['month']            = df['Date'].dt.month
df['days_since_start'] = (df['Date'] - df['Date'].min()).dt.days
df['desc1_word_count'] = df['desc1_clean_bert'].fillna('').apply(lambda x: len(x.split()))
df['desc2_word_count'] = df['desc2_clean_bert'].fillna('').apply(lambda x: len(x.split()))
df['notes_char_count'] = df['notes_clean'].fillna('').str.len()
df['notes_has_content']= (df['notes_char_count'] > 50).astype(int)
df['has_problem_desc'] = df['notes_problem_desc'].notna().astype(int)
df['has_diagnostic']   = df['notes_diagnostic'].notna().astype(int)
# NEW: D2 longer than D1 means case was revised/expanded
df['desc_d2_longer']   = (df['desc2_word_count'] > df['desc1_word_count']).astype(int)
df['desc_len_diff']    = df['desc2_word_count'] - df['desc1_word_count']

# Severity keywords (EXTENDED list vs v1)
SEVERITY_KW = [
    'patient','injury','death','harm','adverse','incident','complaint','legal',
    'regulatory','recall','safety','malfunction','failure','critical','urgent',
    'emergency','shock','pacing','deliver','defibrillat','cardioversion',
    'life','sustain','serious','fatal','abort','spark','burn','smoke',
    # NEW clinical terms from false negative analysis
    'tachycardia','arrhythmia','cardiac','ventricular','fibrillation',
    'stent','catheter','energy','investigation','reportable','radiation',
    'atrial','implant','perforation','tamponade','embolism',
]
full_text = (
    df['desc1_clean_bert'].fillna('') + ' ' +
    df['notes_problem_desc'].fillna('') + ' ' +
    df['notes_clean'].fillna('')
).str.lower()
for kw in SEVERITY_KW:
    df[f'kw_{kw}'] = full_text.str.contains(kw, regex=False).astype(int)

# Compound signals
df['sig_shock_not_delivered'] = (
    full_text.str.contains(r'shock.{0,20}(not|fail|did not|unable)', regex=True) |
    full_text.str.contains('did not deliver', regex=False)
).astype(int)
df['sig_pacing_failure']  = full_text.str.contains(r'pacing.{0,15}fail|fail.{0,15}pac', regex=True).astype(int)
df['sig_patient_impact']  = full_text.str.contains(r'patient.{0,20}(harm|injur|affect|impact)', regex=True).astype(int)
df['sig_critical_fail']   = full_text.str.contains(r'(life|sustain|critical).{0,20}fail|fail.{0,20}(life|sustain|critical)', regex=True).astype(int)
# NEW compound signals from FN analysis
df['sig_ntf_patient']     = (
    full_text.str.contains('no trouble found', regex=False) &
    full_text.str.contains('patient', regex=False)
).astype(int)
df['sig_energy_low']      = full_text.str.contains(r'energy.{0,15}low|low.{0,15}energy', regex=True).astype(int)
df['sig_tachycardia_sw']  = (
    full_text.str.contains('tachycardia', regex=False) &
    full_text.str.contains('software', regex=False)
).astype(int)

print('Target distribution:')
print(df['target'].value_counts(dropna=False))
print(f'Total columns: {len(df.columns)}')

/tmp/ipykernel_1357049/1586197476.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  full_text.str.contains(r'shock.{0,20}(not|fail|did not|unable)', regex=True) |
/tmp/ipykernel_1357049/1586197476.py:43: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['sig_patient_impact']  = full_text.str.contains(r'patient.{0,20}(harm|injur|affect|impact)', regex=True).astype(int)
/tmp/ipykernel_1357049/1586197476.py:44: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['sig_critical_fail']   = full_text.str.contains(r'(life|sustain|critical).{0,20}fail|fail.{0,20}(life|sustain|critical)', regex=True).astype(int)


Target distribution:
target
0.0    3849
1.0     133
NaN      17
Name: count, dtype: int64
Total columns: 82


In [11]:
# ── Sanity check: before vs after on key corruption types ─────────────────
print('=== CORRUPTION FIX VALIDATION ===')
fixed_notes = df['notes_clean'].fillna('').astype(str)
orig_notes  = df['Notes'].fillna('').astype(str)

checks = [
    ('Cutomer',    'Customer'),
    ('failue',     'failure'),
    ('uLL',        'error'),
    ('touhscreen', 'touchscreen'),
]
for broken, fixed in checks:
    orig_count  = int(orig_notes.str.count(broken).sum())
    fixed_count = int(fixed_notes.str.count(broken).sum())
    new_count   = int(fixed_notes.str.count(fixed).sum())
    print(f'  {broken:15s} → {fixed:12s} | before: {orig_count:5d}  after broken: {fixed_count:4d}  new fixed: {new_count:5d}')

print()
print('=== SAMPLE: desc1_clean_bert ===')
for i in df.sample(6, random_state=42).index:
    print(f'  ORIG : {str(df.loc[i,"Description 1"])[:80]}')
    print(f'  CLEAN: {str(df.loc[i,"desc1_clean_bert"])[:80]}')
    print()

=== CORRUPTION FIX VALIDATION ===
  Cutomer         → Customer     | before:  8365  after broken:   85  new fixed:  8588
  failue          → failure      | before:  1309  after broken:   65  new fixed:  1295
  uLL             → error        | before:  5372  after broken:  744  new fixed:  3021
  touhscreen      → touchscreen  | before:   213  after broken:   18  new fixed:   199

=== SAMPLE: desc1_clean_bert ===
  ORIG : M0A / u00/ Failed Ops check. -
  CLEAN: m0a u00 failed ops check

  ORIG : 00he defibrillator one of the paddles does not work.
  CLEAN: he defibrillator one of the paddles does not work

  ORIG : System Failure
  CLEAN: system failure

  ORIG : Image Storage failure
  CLEAN: image storage failure

  ORIG : Fleuvision PC startup problem
  CLEAN: fleuvision pc startup problem

  ORIG : QA00000 - self-test failed
  CLEAN: qa self test failed



In [10]:
df.to_excel(OUTPUT_PATH, index=False)
print(f'Saved → {OUTPUT_PATH}  |  Shape: {df.shape}')
print('\nNew columns vs v1:')
new_cols = ['desc2_clean_bert','se_flag','regulatory_flag','desc_d2_longer',
            'desc_len_diff','sig_ntf_patient','sig_energy_low','sig_tachycardia_sw']
print('  ' + ', '.join(new_cols))

Saved → DATA_cleaned_v2.xlsx  |  Shape: (3999, 82)

New columns vs v1:
  desc2_clean_bert, se_flag, regulatory_flag, desc_d2_longer, desc_len_diff, sig_ntf_patient, sig_energy_low, sig_tachycardia_sw
